In [2]:
!pip install pandas openpyxl

In [8]:
import pandas as pd

df = pd.read_excel('Х5.xlsx')

df.head()

,new_id,Месяц,Трафик,Средний чек,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,10,59662,823.060390,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
1,0,5,56674,859.361975,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
2,0,1,51488,763.937766,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
3,0,6,56693,836.362309,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
4,0,7,58128,845.257709,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0


In [48]:
df_cut = df[['new_id', 'Трафик', 'Средний чек', 'Населенный пункт', 'Регион', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']].copy()

In [49]:
df_cut

,new_id,Трафик,Средний чек,Населенный пункт,Регион,"Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,59662,823.060390,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0
1,0,56674,859.361975,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0
2,0,51488,763.937766,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0
3,0,56693,836.362309,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0
4,0,58128,845.257709,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...
256718,21742,51676,1167.101083,Октябрьский рп,Волгоградская обл,1,1,0,0,1,0
256719,21742,51516,1252.914118,Октябрьский рп,Волгоградская обл,1,1,0,0,1,0
256720,21742,49593,1130.823998,Октябрьский рп,Волгоградская обл,1,1,0,0,1,0
256721,21742,52115,1461.929305,Октябрьский рп,Волгоградская обл,1,1,0,0,1,0


In [50]:
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import GroupKFold

In [51]:
gkf = GroupKFold(n_splits=5)

In [56]:
df_cut['Населенный пункт_fold_traffic'] = np.nan
df_cut['Регион_fold_traffic'] = np.nan
df_cut['Населенный пункт_fold_average_bill'] = np.nan
df_cut['Регион_fold_average_bill'] = np.nan

In [57]:
for train_id, validation_id in gkf.split(df_cut, groups=df_cut['new_id']):
    
    X_train = df_cut.iloc[train_id][['Населенный пункт', 'Регион']]
    X_validation = df_cut.iloc[validation_id][['Населенный пункт', 'Регион']]
    y_train_traffic = df_cut.iloc[train_id]['Трафик']
    y_train_average_bill = df_cut.iloc[train_id]['Средний чек']
    
    TE_traffic = TargetEncoder(cols = ['Населенный пункт', 'Регион'], smoothing = 20)
    TE_traffic.fit(X_train, y_train_traffic)
    val_traffic = TE_traffic.transform(X_validation)
    
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Населенный пункт_fold_traffic')] = val_traffic['Населенный пункт']
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Регион_fold_traffic')] = val_traffic['Регион']
    
    TE_average_bill = TargetEncoder(cols = ['Населенный пункт', 'Регион'], smoothing = 20)
    TE_average_bill.fit(X_train, y_train_average_bill)
    val_average_bill = TE_average_bill.transform(X_validation)
    
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Населенный пункт_fold_average_bill')] = val_average_bill['Населенный пункт']
    df_cut.iloc[validation_id, df_cut.columns.get_loc('Регион_fold_average_bill')] = val_average_bill['Регион']

In [60]:
df_cut.head(5)

,new_id,Трафик,Средний чек,Населенный пункт,Регион,"Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Населенный пункт_fold_traffic,Регион_fold_traffic,Населенный пункт_fold_average_bill,Регион_fold_average_bill
0,0,59662,823.060390,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0,58840.475518,57741.947383,965.732862,880.841244
1,0,56674,859.361975,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0,58840.475518,57741.947383,965.732862,880.841244
2,0,51488,763.937766,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0,58840.475518,57741.947383,965.732862,880.841244
3,0,56693,836.362309,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0,58840.475518,57741.947383,965.732862,880.841244
4,0,58128,845.257709,Кавказская ст-ца,Краснодарский край,0,6,0,0,2,0,58840.475518,57741.947383,965.732862,880.841244


In [61]:
df_cut_traffic = df_cut[['new_id', 'Трафик', 'Средний чек', 'Населенный пункт_fold_traffic', 'Регион_fold_traffic', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']]

In [62]:
df_cut_traffic

,new_id,Трафик,Средний чек,Населенный пункт_fold_traffic,Регион_fold_traffic,"Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,59662,823.060390,58840.475518,57741.947383,0,6,0,0,2,0
1,0,56674,859.361975,58840.475518,57741.947383,0,6,0,0,2,0
2,0,51488,763.937766,58840.475518,57741.947383,0,6,0,0,2,0
3,0,56693,836.362309,58840.475518,57741.947383,0,6,0,0,2,0
4,0,58128,845.257709,58840.475518,57741.947383,0,6,0,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...
256718,21742,51676,1167.101083,52384.445118,54212.168119,1,1,0,0,1,0
256719,21742,51516,1252.914118,52384.445118,54212.168119,1,1,0,0,1,0
256720,21742,49593,1130.823998,52384.445118,54212.168119,1,1,0,0,1,0
256721,21742,52115,1461.929305,52384.445118,54212.168119,1,1,0,0,1,0


In [64]:
df_cut_average_bill = df_cut[['new_id', 'Трафик', 'Средний чек', 'Населенный пункт_fold_average_bill', 'Регион_fold_average_bill', 'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)']]

In [65]:
df_cut_average_bill

,new_id,Трафик,Средний чек,Населенный пункт_fold_average_bill,Регион_fold_average_bill,"Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,59662,823.060390,965.732862,880.841244,0,6,0,0,2,0
1,0,56674,859.361975,965.732862,880.841244,0,6,0,0,2,0
2,0,51488,763.937766,965.732862,880.841244,0,6,0,0,2,0
3,0,56693,836.362309,965.732862,880.841244,0,6,0,0,2,0
4,0,58128,845.257709,965.732862,880.841244,0,6,0,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...
256718,21742,51676,1167.101083,930.736809,792.641594,1,1,0,0,1,0
256719,21742,51516,1252.914118,930.736809,792.641594,1,1,0,0,1,0
256720,21742,49593,1130.823998,930.736809,792.641594,1,1,0,0,1,0
256721,21742,52115,1461.929305,930.736809,792.641594,1,1,0,0,1,0
